# AppWorld Spotify API Investigation

Testing task `82e2fac_1` to investigate whether the Spotify API is actually wrong for AppWorld or if there's an implementation issue.


In [1]:
from appworld import AppWorld

# Load the specific task mentioned in the outputs
task_id = "82e2fac_1"
world = AppWorld(task_id)
apis = world.apis

print(f"Task ID: {task_id}")
print(f"World initialized: {world}")
print(f"APIs available: {apis}")


Task ID: 82e2fac_1
World initialized: <appworld.environment.AppWorld object at 0x7e98b22f6d10>
APIs available: ApiCollection({'admin': _CustomErrorMessageMunch({'credit_to_card': <function _wrap_api_request.<locals>._api_request at 0x7e9893f3b060>, 'debit_from_card': <function _wrap_api_request.<locals>._api_request at 0x7e9893f3b100>}), 'amazon': _CustomErrorMessageMunch({'add_address': <function _wrap_api_request.<locals>._api_request at 0x7e9893f3b1a0>, 'add_gift_wrapping_to_product': <function _wrap_api_request.<locals>._api_request at 0x7e9893f3b240>, 'add_payment_card': <function _wrap_api_request.<locals>._api_request at 0x7e9893f3b2e0>, 'add_product_to_browsing_history': <function _wrap_api_request.<locals>._api_request at 0x7e9893f3b380>, 'add_product_to_cart': <function _wrap_api_request.<locals>._api_request at 0x7e9893f3b420>, 'add_product_to_wish_list': <function _wrap_api_request.<locals>._api_request at 0x7e9893f3b4c0>, 'apply_promo_code_to_cart': <function _wrap_api_req

In [2]:
# Check what apps are available
print("Available apps:")
print(apis.api_docs.show_app_descriptions())


Available apps:
[{'name': 'api_docs', 'description': 'An app to search and explore API documentation.'}, {'name': 'supervisor', 'description': "An app to access supervisor's personal information, account credentials, addresses, payment cards, and manage the assigned task."}, {'name': 'amazon', 'description': 'An online shopping app to buy products and manage orders, returns, etc.'}, {'name': 'phone', 'description': 'An app to find and manage contact information for friends, family members, etc., send and receive messages, and manage alarms.'}, {'name': 'file_system', 'description': 'A file system app to create and manage files and folders.'}, {'name': 'spotify', 'description': 'A music streaming app to stream songs and manage song, album and playlist libraries.'}, {'name': 'venmo', 'description': 'A social payment app to send, receive and request money to and from others.'}, {'name': 'gmail', 'description': 'An email app to draft, send, receive, and manage emails.'}, {'name': 'splitwis

In [3]:
# Check Spotify APIs specifically
print("Spotify APIs:")
print(apis.api_docs.show_api_descriptions(app_name='spotify'))


Spotify APIs:
[{'name': 'show_account', 'description': 'Show your account information. Unlike show_profile, this includes private information.'}, {'name': 'signup', 'description': 'Sign up to create account.'}, {'name': 'delete_account', 'description': 'Delete your account.'}, {'name': 'update_account_name', 'description': 'Update your first or last name in the account profile.'}, {'name': 'login', 'description': 'Login to your account.'}, {'name': 'logout', 'description': 'Logout from your account.'}, {'name': 'send_verification_code', 'description': 'Send account verification code to your email address.'}, {'name': 'verify_account', 'description': 'Verify your account using the verification code sent to your email address.'}, {'name': 'send_password_reset_code', 'description': 'Send password reset code to your email address.'}, {'name': 'reset_password', 'description': 'Reset your password using the password reset code sent to your email address.'}, {'name': 'show_profile', 'descript

In [4]:
# Get the login API specification
print("Spotify login API spec:")
print(apis.api_docs.show_api_doc(app_name='spotify', api_name='login'))


Spotify login API spec:
{'app_name': 'spotify', 'api_name': 'login', 'path': '/spotify/auth/token', 'method': 'POST', 'description': 'Login to your account.', 'parameters': [{'name': 'username', 'type': 'string', 'required': True, 'description': 'Your account email.', 'default': None, 'constraints': []}, {'name': 'password', 'type': 'string', 'required': True, 'description': 'Your account password.', 'default': None, 'constraints': []}], 'response_schemas': {'success': {'access_token': 'string', 'token_type': 'string'}, 'failure': {'message': 'string'}}}


In [5]:
# Get supervisor credentials
print("Supervisor passwords:")
passwords = apis.supervisor.show_account_passwords()
print(passwords)


Supervisor passwords:
[{'account_name': 'amazon', 'password': 'FJRd9=B'}, {'account_name': 'file_system', 'password': 'DqE8={8'}, {'account_name': 'gmail', 'password': 'r^2p&]H'}, {'account_name': 'phone', 'password': 'QAEZ+BF'}, {'account_name': 'simple_note', 'password': 'RluCyXn'}, {'account_name': 'splitwise', 'password': 'u4uIy!w'}, {'account_name': 'spotify', 'password': 'qge1k1L'}, {'account_name': 'todoist', 'password': '&Jf9F11'}, {'account_name': 'venmo', 'password': 'I!)k(T8'}]


In [7]:
# Get user information
print("User information:")
user_info = apis.supervisor.show_user_info()
print(user_info)


User information:


HTTPException: 422: No API named 'show_user_info' found in the supervisor app.

In [6]:
# Try to login with the credentials
print("Attempting Spotify login...")

# Extract Spotify password
spotify_password = None
for account_password in passwords:
    if account_password["account_name"] == "spotify":
        spotify_password = account_password["password"]
        break

print(f"Spotify password found: {spotify_password}")
print(f"User email: {user_info.get('email', 'Not found')}")

# Attempt login
try:
    login_result = apis.spotify.login(
        username=user_info.get('email', 'test@example.com'), 
        password=spotify_password
    )
    print(f"Login successful: {login_result}")
except Exception as e:
    print(f"Login failed: {e}")


Attempting Spotify login...
Spotify password found: qge1k1L


NameError: name 'user_info' is not defined

In [ ]:
# If login works, test other Spotify APIs
if 'login_result' in locals() and login_result:
    access_token = login_result.get('access_token')
    print(f"Testing with access token: {access_token}")
    
    # Test playlist library API
    try:
        playlists = apis.spotify.show_playlist_library(access_token=access_token)
        print(f"Playlists retrieved: {len(playlists) if playlists else 0}")
        print(f"First few playlists: {playlists[:3] if playlists else 'None'}")
    except Exception as e:
        print(f"Playlist library API failed: {e}")
else:
    print("Cannot test other APIs - login failed")
